# Thí nghiệm re-ID trên cả hiệp (Kaggle)

Chạy pipeline minimap trên **cả 1 hiệp** (`1_720p.mp4`, ~52 phút, 30 fps), rồi tính embedding của 3 model re-ID
(MSMT17, Deep-EIoU SportsMOT, SoccerNet-ReID) và đo bằng `tools/reid_bench.py` + `tools/link_bench.py`.
Kết quả được nén thành `/kaggle/working/reid_half1.zip`, tải về rồi giải nén vào `outputs/` ở máy local.

**Settings:**
- *Accelerator*: **GPU T4** (tránh P100).
- *Internet*: **On** (clone repo, tải weights).
- *Add Input*: Kaggle Dataset chứa `1_720p.mp4`.
- Chạy bằng **Save Version → Save & Run All (Commit)**: chạy nền tối đa 12 giờ, không bị ngắt khi đóng tab.
  Nên test trước ở chế độ interactive với `MAX_FRAMES = 1500` (cell 4) để chắc mọi thứ chạy.


In [ ]:
# 1. Clone repo vào /tmp (để /kaggle/working chỉ chứa kết quả) + cài đặt + tải weights
REPO_URL = "https://github.com/nvdat16/SoccerPipeline.git"
!git clone -q $REPO_URL /tmp/SoccerPipeline
%cd /tmp/SoccerPipeline
!bash scripts/setup.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Checkpoint SoccerNet-ReID OSNet x1.0 (sn-reid baseline, ~1 GB vì classifier 161k lớp; loader tự bỏ classifier)
import os
if not os.path.exists("models/osnet_x1_0_soccernet_reid.pt"):
    !gdown -q 1To0Ww6_HxU2ITAlb4kQEgYExV-orwit8 -O models/osnet_x1_0_soccernet_reid.pt
!ls -la models/

In [ ]:
# 3. Video input
import glob
videos = sorted(glob.glob("/kaggle/input/**/1_720p.mp4", recursive=True)) or \
         sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
print("\n".join(videos) or "Chưa có video — Add Input dataset chứa 1_720p.mp4")
VIDEO = videos[0]
print("=> VIDEO =", VIDEO)
os.environ["VIDEO"] = VIDEO

In [ ]:
# 4. Pipeline minimap (không render video). MAX_FRAMES = 0 là cả hiệp; đặt 1500 để test nhanh trước.
import os
from pathlib import Path
MAX_FRAMES = 0
OUT_ROOT = "/kaggle/working/outputs/half1"
os.environ["RUN"] = RUN = f"{OUT_ROOT}/{Path(VIDEO).stem}"
!MAX_FRAMES=$MAX_FRAMES bash scripts/run_minimap.sh "$VIDEO" $OUT_ROOT --no-video
!ls -la $RUN

In [ ]:
%%bash
# 5. Embedding re-ID cho cả 3 model (mọi box player/GK >= 40px, mỗi 2 frame — giống run.py)
python tools/reid_bench.py embed "$RUN" --video "$VIDEO" --offset 0 \
    --weights models/osnet_ain_x1_0_msmt17.pt models/osnet_x1_0_sportsmot_deepeiou.pt models/osnet_x1_0_soccernet_reid.pt

In [ ]:
%%bash
# 6. Đo (CPU, chỉ đọc CSV + .npz). Log lưu vào $RUN/bench/
M="osnet_ain_x1_0_msmt17 osnet_x1_0_sportsmot_deepeiou osnet_x1_0_soccernet_reid"
mkdir -p "$RUN/bench"
python tools/reid_bench.py probe "$RUN" --emb "$RUN"/emb_*.npz 2>&1 | tee "$RUN/bench/02_probe.log"
for mg in 0.0 0.03 0.06 0.1 0.15; do
  python tools/reid_bench.py probe "$RUN" --margin $mg --emb "$RUN"/emb_*.npz
done 2>&1 | tee "$RUN/bench/05_probe_margin_sweep.log"
{ echo "=== không re-ID"; python tools/link_bench.py "$RUN"
  for m in $M; do echo "=== $m"; python tools/link_bench.py "$RUN" --emb "$RUN/emb_$m.npz"; done
} 2>&1 | tee "$RUN/bench/03_link_bench.log"
{ python tools/relink.py "$RUN" --out "$RUN/relink_noreid"
  for m in $M; do python tools/relink.py "$RUN" --emb "$RUN/emb_$m.npz" --out "$RUN/relink_$m"; done
} 2>&1 | tee "$RUN/bench/04_relink_real.log"

In [ ]:
%%bash
# 7. Nén kết quả để tải về (Output -> reid_half1.zip)
cd /kaggle/working/outputs && zip -qr /kaggle/working/reid_half1.zip half1
ls -la /kaggle/working/reid_half1.zip